# Which Way Round Does Your KL Go?

**Topic:** KL divergence &nbsp;·&nbsp; **Difficulty:** Medium

`KL(P||Q)` and `KL(Q||P)` are different numbers and, more importantly, different
*objectives*. Minimizing the forward direction makes `Q` cover every mode of `P`
even if that means putting mass where `P` has none. The reverse direction
makes `Q` lock onto one mode and ignore the rest.

Fit a single Gaussian to a two-mode target both ways and see the two answers.

## The data (created in the next cell)

- `p`: a bimodal target on a grid of 2000 points, a mixture of two Gaussians
  at `-3` and `+3`.
- `grid`: the evaluation points.
- Candidate fits: single Gaussians over a grid of means and standard deviations.

## Your task

With **NumPy only**:

1. Compute both divergences on the grid (discrete sums, with the convention
   `0 log 0 = 0`), for every candidate `(mu, sigma)`. Normalize each candidate
   `q` so it sums to 1 over the grid, the same convention as `p`.
2. `fwd_best`: the `(mu, sigma)` minimizing `KL(p || q)` (mass-covering).
3. `rev_best`: the `(mu, sigma)` minimizing `KL(q || p)` (mode-seeking).
4. At the forward optimum, verify the identity `H(p, q) = H(p) + KL(p||q)` by
   computing both sides. You will report the absolute gap as `identity_gap`
   in step 5.
5. Set `answers` to a dict with `"fwd_mu"`, `"fwd_sigma"`, `"rev_mu"`,
   `"rev_sigma"`, and `"identity_gap"`, the absolute error in the
   cross-entropy identity.

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
checks the forward fit is much wider than the reverse fit and the identity
holds.

In [ ]:
import numpy as np

grid = np.linspace(-10, 10, 2000)
dx = grid[1] - grid[0]

def gauss(x, mu, sd):
    return np.exp(-0.5 * ((x - mu) / sd) ** 2) / (sd * np.sqrt(2 * np.pi))

p = 0.5 * gauss(grid, -3.0, 0.7) + 0.5 * gauss(grid, 3.0, 0.7)
p = p / p.sum()

mus = np.linspace(-4, 4, 81)
sigmas = np.linspace(0.3, 6.0, 115)
print(f"target is bimodal at -3 and +3. Searching {len(mus)}x{len(sigmas)} Gaussians")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the target distribution and candidate grids created above, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("fwd_mu", [], 0.0, 0.0, 1e-06),
    ("fwd_sigma", [], 3.1, -0.5190578459249474, 1e-06),
    ("rev_sigma", [], 0.7, -0.1172066103701494, 1e-06),
    ("identity_gap", [], 1.7763568394002505e-15, -2.9742966279133604e-16, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "rev_mu" in answers:
    _mode = abs(abs(float(answers["rev_mu"])) - 3.0) <= 1e-06
    print(f"  {'rev_mu |mu| = 3':<24} {'ok' if _mode else 'wrong'}")
    _ok = _ok and _mode

if "fwd_sigma" in answers:
    _wide = answers["fwd_sigma"] > 2 * answers["rev_sigma"]
    _id = answers["identity_gap"] < 1e-9
    print(f"  {'forward covers, reverse locks':<24} {'ok' if _wide else 'wrong'}  "
          f"(sigma {answers['fwd_sigma']:.2f} vs {answers['rev_sigma']:.2f})")
    print(f"  {'H(p,q) = H(p) + KL':<24} {'ok' if _id else 'wrong'}")
    _ok = _ok and _wide and _id

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

Forward KL `sum p log(p/q)` is infinite wherever `p > 0` and `q = 0`, so the fit
is forced to stretch over both modes, so it lands wide and centred between them,
putting substantial mass in the valley where the target has almost none. Reverse
KL `sum q log(q/p)` penalizes `q > 0` where `p` is small, so it collapses onto one
mode and ignores the other entirely. Maximum likelihood is forward KL, which is
why likelihood-trained generative models blur. Variational inference is
reverse KL, which is why it under-covers.

```python
import numpy as np

eps = 1e-300
best_f = (np.inf, None)
best_r = (np.inf, None)
for mu in mus:
    for sd in sigmas:
        q = gauss(grid, mu, sd)
        q = q / q.sum()
        fwd = float(np.sum(np.where(p > 0, p * np.log((p + eps) / (q + eps)), 0.0)))
        rev = float(np.sum(np.where(q > 0, q * np.log((q + eps) / (p + eps)), 0.0)))
        if fwd < best_f[0]:
            best_f = (fwd, (mu, sd))
        if rev < best_r[0]:
            best_r = (rev, (mu, sd))

fmu, fsd = best_f[1]
rmu, rsd = best_r[1]

qf = gauss(grid, fmu, fsd)
qf = qf / qf.sum()
H_p = float(-np.sum(np.where(p > 0, p * np.log(p + eps), 0.0)))
H_pq = float(-np.sum(np.where(p > 0, p * np.log(qf + eps), 0.0)))

answers = {
    "fwd_mu": float(fmu),
    "fwd_sigma": float(fsd),
    "rev_mu": float(rmu),
    "rev_sigma": float(rsd),
    "identity_gap": float(abs(H_pq - (H_p + best_f[0]))),
}
```